# 2026 COMP90042 Project — Retrieval Branch v5

基于 v3。只研究粗排/精排，不做 classifier 调参。固定 BM25 candidate K=500；不重复 top1000 和 stance。一次运行完成 BM25 参数 sweep、BM25 variant union、每个候选策略的 CE reranker 训练与 selector tuning。

In [1]:
# Optional dependency installation. Safe in Colab; usually skipped locally if packages already exist.
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "bm25s": "bm25s",
    "transformers": "transformers",
    "accelerate": "accelerate",
    "psutil": "psutil",
}

for pip_name, import_name in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

# 1. Data and config

In [2]:

from pathlib import Path
import json
import random
import time
import pickle
import re
import gc
from collections import Counter

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

# -------------------------
# Retrieval branch config
# -------------------------
SEED = 42
FAST_DEV_MODE = False  # True = quick smoke test; False = full project run

DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs_notebook_retrieval_branch")
CACHE_DIR = OUTPUT_DIR / "cache"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
CACHE_DIR.mkdir(exist_ok=True, parents=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


def safe_name(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "-", str(value)).strip("-")


# Fixed by prior experiments: do not repeat top1000, keep candidate pool at 500.
BM25_CANDIDATE_K = 500 if not FAST_DEV_MODE else 50

# Keep final evidence selector sweep because it is cheap and directly controls retrieval F.
FIXED_K_GRID = [2, 3, 4, 5]
THRESHOLD_GRID = [round(float(x), 2) for x in np.arange(0.02, 0.52, 0.02)] + [0.60, 0.70, 0.80]
RELATIVE_LOGIT_DELTA_GRID = [0.50, 0.75, 1.00, 1.50, 2.00, 3.00]
MIN_FINAL_K = 1
MAX_FINAL_K = 5
FINAL_RETRIEVAL_POLICY = "prefer_dynamic"
DYNAMIC_RETRIEVAL_TOLERANCE = 0.02

# Cross-encoder reranker config.
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
RERANKER_MAX_LEN = 256
RERANKER_BATCH_SIZE = 32 if not FAST_DEV_MODE else 8
RERANKER_EVAL_BATCH_SIZE = 128 if not FAST_DEV_MODE else 16
RERANKER_EPOCHS = 3 if not FAST_DEV_MODE else 1
RERANKER_LR = 1e-5
NEGATIVES_PER_POSITIVE = 4 if not FAST_DEV_MODE else 2
HARD_NEGATIVE_POOL = min(200, BM25_CANDIDATE_K)
WEIGHT_DECAY = 0.01

# Cache/checkpoint switches. Leave False for normal reruns.
SAVE_BM25_INDEX = True
SAVE_MODEL_CHECKPOINTS = True
FORCE_REBUILD_BM25_INDEX = False
FORCE_RECOMPUTE_BM25_CANDIDATES = False
FORCE_RERANKER_RETRAIN = False
FORCE_RESCORE_CE = False

# Stage 1: cheap BM25 parameter sweep. These are all evaluated by candidate-pool recall@500.
# The default bm25s setting is approximately k1=1.5, b=0.75.
BM25_PARAM_GRID = [
    {"name": "baseline_stop_en", "stopwords": "en", "k1": 1.5, "b": 0.75},
    {"name": "low_b_stop_en", "stopwords": "en", "k1": 1.5, "b": 0.30},
    {"name": "mid_b_stop_en", "stopwords": "en", "k1": 1.5, "b": 0.50},
    {"name": "high_b_stop_en", "stopwords": "en", "k1": 1.5, "b": 0.90},
    {"name": "low_k1_stop_en", "stopwords": "en", "k1": 0.8, "b": 0.75},
    {"name": "mid_k1_stop_en", "stopwords": "en", "k1": 1.2, "b": 0.75},
    {"name": "high_k1_stop_en", "stopwords": "en", "k1": 2.0, "b": 0.75},
    {"name": "low_k1_low_b_stop_en", "stopwords": "en", "k1": 0.8, "b": 0.30},
    {"name": "mid_k1_low_b_stop_en", "stopwords": "en", "k1": 1.2, "b": 0.30},
    {"name": "baseline_no_stop", "stopwords": None, "k1": 1.5, "b": 0.75},
    {"name": "low_b_no_stop", "stopwords": None, "k1": 1.5, "b": 0.30},
]

# Stage 2: expensive reranker stage. We do not train rerankers for every BM25 setting.
# Instead: baseline + top-N single BM25 configs from Stage 1 + RRF union configs.
RERANKER_STAGE_TOP_SINGLE_CONFIGS = 3 if not FAST_DEV_MODE else 1
RUN_RERANKER_STAGE = True
BM25_RRF_K = 60

CACHE_VERSION = "retrieval_branch_v5_from_v3_top500"


def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


set_seed(SEED)


Device: cuda


D:\_Search\_Study\COMP90042-NLP\A3_Group\COMP90042_2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


train_claims = load_json(DATA_DIR / "train-claims.json")
dev_claims = load_json(DATA_DIR / "dev-claims.json")
test_claims = load_json(DATA_DIR / "test-claims-unlabelled.json")
evidence = load_json(DATA_DIR / "evidence.json")

if FAST_DEV_MODE:
    train_claims = dict(list(train_claims.items())[:80])
    dev_claims = dict(list(dev_claims.items())[:30])
    test_claims = dict(list(test_claims.items())[:30])

print(f"train:    {len(train_claims)}")
print(f"dev:      {len(dev_claims)}")
print(f"test:     {len(test_claims)}")
print(f"evidence: {len(evidence)}")

train:    1228
dev:      154
test:     153
evidence: 1208827


In [4]:
LABELS = ["SUPPORTS", "REFUTES", "NOT_ENOUGH_INFO", "DISPUTED"]

# Lightweight EDA used to justify later choices.
label_dist = Counter(c["claim_label"] for c in train_claims.values())
print("Train label distribution:")
for lbl in LABELS:
    cnt = label_dist[lbl]
    print(f"  {lbl:20s} {cnt:5d} ({cnt / len(train_claims):6.2%})")

gt_counts = [len(c["evidences"]) for c in train_claims.values()]
print("\nGround-truth evidence count per train claim:")
print("  min=", min(gt_counts), "max=", max(gt_counts), "mean=", round(float(np.mean(gt_counts)), 3))
print("  exact counts:", sorted(Counter(gt_counts).items()))

Train label distribution:
  SUPPORTS               519 (42.26%)
  REFUTES                199 (16.21%)
  NOT_ENOUGH_INFO        386 (31.43%)
  DISPUTED               124 (10.10%)

Ground-truth evidence count per train claim:
  min= 1 max= 5 mean= 3.357
  exact counts: [(1, 210), (2, 223), (3, 191), (4, 127), (5, 477)]


In [5]:
# Official-style metrics. These mirror eval.py's logic and let us tune inside the notebook.
def evidence_f1_for_claim(pred_eids, gold_eids):
    pred_eids = list(pred_eids)
    gold_eids = list(gold_eids)
    if len(pred_eids) == 0:
        return 0.0
    pred_set = set(pred_eids)
    correct = sum(1 for eid in gold_eids if eid in pred_set)
    if correct == 0:
        return 0.0
    precision = correct / len(pred_eids)
    recall = correct / len(gold_eids)
    return 2 * precision * recall / (precision + recall)


def evaluate_submission(predictions, gold_claims, verbose=True):
    f_scores = []
    correct_labels = 0
    total = 0
    for cid, gold in gold_claims.items():
        pred = predictions[cid]
        f_scores.append(evidence_f1_for_claim(pred["evidences"], gold["evidences"]))
        correct_labels += int(pred["claim_label"] == gold["claim_label"])
        total += 1
    F = float(np.mean(f_scores))
    A = correct_labels / total
    H = 0.0 if (F + A) == 0 else 2 * F * A / (F + A)
    if verbose:
        print(f"Evidence Retrieval F-score (F)    = {F:.6f}")
        print(f"Claim Classification Accuracy (A) = {A:.6f}")
        print(f"Harmonic Mean of F and A          = {H:.6f}")
    return {"F": F, "A": A, "H": H}


def evaluate_retrieval_only(retrieval, gold_claims):
    return float(
        np.mean(
            [
                evidence_f1_for_claim(retrieval[cid], claim["evidences"])
                for cid, claim in gold_claims.items()
            ]
        )
    )


def majority_label(claims):
    return Counter(c["claim_label"] for c in claims.values()).most_common(1)[0][0]


def validate_retrieval_coverage(claims_dict, retrieval, split_name="split"):
    """Fail early with a helpful message if retrieval is incomplete or empty."""
    missing_cids = [cid for cid in claims_dict if cid not in retrieval]
    if missing_cids:
        raise KeyError(f"{split_name}: retrieval missing {len(missing_cids)} claim ids, e.g. {missing_cids[:3]}")

    empty_cids = [cid for cid in claims_dict if len(retrieval[cid]) == 0]
    if empty_cids:
        raise ValueError(f"{split_name}: retrieval has empty evidence lists, e.g. {empty_cids[:3]}")

    bad_eids = []
    for cid in claims_dict:
        for eid in retrieval[cid]:
            if eid not in evidence:
                bad_eids.append((cid, eid))
                if len(bad_eids) >= 3:
                    break
        if len(bad_eids) >= 3:
            break
    if bad_eids:
        raise KeyError(f"{split_name}: retrieval contains unknown evidence ids, e.g. {bad_eids}")


def build_predictions(claims, retrieval, label_predictions=None, default_label=None):
    if label_predictions is None:
        assert default_label is not None
        label_predictions = {cid: default_label for cid in claims.keys()}
    out = {}
    fallback_eid = next(iter(evidence.keys()))
    for cid in claims.keys():
        eids = list(retrieval.get(cid, []))
        if len(eids) == 0:
            # Assignment requires at least one evidence. This fallback should rarely trigger.
            eids = [fallback_eid]
        out[cid] = {"claim_label": label_predictions[cid], "evidences": eids}
    return out


def write_predictions(predictions, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    # ensure_ascii=True keeps the file readable even when Windows uses a non-UTF-8 default encoding.
    with open(path, "w", encoding="utf-8") as f:
        json.dump(predictions, f, indent=2, ensure_ascii=True)
    print("Wrote", path)


## Stage 1: BM25 parameter sweep and variant candidate pools

In [6]:

import bm25s

# bm25s may print "resource module not available on Windows". It is harmless.
evidence_ids = list(evidence.keys())
evidence_texts = [evidence[eid] for eid in evidence_ids]


def spec_name(spec):
    # Keep union names short to avoid Windows path length issues.
    if spec.get("type") == "rrf_union":
        return safe_name(f"rrf_{spec['name']}")
    stop = "nostop" if spec.get("stopwords") is None else f"stop-{spec.get('stopwords')}"
    return safe_name(f"{spec['name']}_{stop}_k1{spec.get('k1', 1.5)}_b{spec.get('b', 0.75)}")


def build_or_load_bm25_index(spec):
    """Build/load one single BM25 index for a lexical view."""
    assert spec.get("type", "single") == "single"
    name = spec_name(spec)
    index_dir = CACHE_DIR / f"bm25s_index_{name}_{len(evidence_ids)}docs"
    retriever = None

    if SAVE_BM25_INDEX and index_dir.exists() and not FORCE_REBUILD_BM25_INDEX:
        try:
            print("Loading BM25 index:", index_dir)
            retriever = bm25s.BM25.load(str(index_dir), load_corpus=False)
        except Exception as exc:
            print("Could not load cached BM25 index; rebuilding. Reason:", repr(exc))
            retriever = None

    if retriever is None:
        print(f"Tokenizing/building BM25 index: {name}")
        t0 = time.time()
        corpus_tokens = bm25s.tokenize(evidence_texts, stopwords=spec.get("stopwords"), stemmer=None)
        retriever = bm25s.BM25(k1=float(spec.get("k1", 1.5)), b=float(spec.get("b", 0.75)))
        retriever.index(corpus_tokens)
        print(f"Built {name} in {time.time() - t0:.1f}s")
        if SAVE_BM25_INDEX:
            try:
                retriever.save(str(index_dir))
            except Exception as exc:
                print("Warning: BM25 index save failed; candidate caches will still work. Reason:", repr(exc))
    return retriever


def retrieve_single_bm25(claims_dict, spec, k=BM25_CANDIDATE_K):
    retriever = build_or_load_bm25_index(spec)
    cids = list(claims_dict.keys())
    queries = [claims_dict[cid]["claim_text"] for cid in cids]
    query_tokens = bm25s.tokenize(queries, stopwords=spec.get("stopwords"), stemmer=None)
    results, scores = retriever.retrieve(query_tokens, k=k)

    out = {}
    for i, cid in enumerate(cids):
        pairs = []
        for j, score in zip(results[i], scores[i]):
            pairs.append((evidence_ids[int(j)], float(score)))
        out[cid] = pairs
    return out


def rrf_fuse(candidate_runs, keep_k=BM25_CANDIDATE_K, rrf_k=BM25_RRF_K):
    """Reciprocal Rank Fusion over multiple {cid: [(eid, score)]} candidate runs."""
    cids = list(candidate_runs[0].keys())
    fused = {}
    for cid in cids:
        scores = {}
        best_raw = {}
        for run in candidate_runs:
            for rank, (eid, raw_score) in enumerate(run[cid], start=1):
                scores[eid] = scores.get(eid, 0.0) + 1.0 / (rrf_k + rank)
                best_raw[eid] = max(best_raw.get(eid, float("-inf")), float(raw_score))
        ranked = sorted(scores, key=lambda eid: (-scores[eid], -best_raw[eid], eid))[:keep_k]
        fused[cid] = [(eid, float(scores[eid])) for eid in ranked]
    return fused


def compute_or_load_candidates(claims_dict, split_name, spec, k=BM25_CANDIDATE_K):
    name = spec_name(spec)
    cache_file = CACHE_DIR / f"{split_name}_candidates_{name}_top{k}.pkl"
    if cache_file.exists() and not FORCE_RECOMPUTE_BM25_CANDIDATES:
        print("Loading cached candidates:", cache_file)
        with open(cache_file, "rb") as f:
            return pickle.load(f)

    print(f"Computing {split_name} candidates for {name} ...")
    if spec.get("type") == "rrf_union":
        runs = [compute_or_load_candidates(claims_dict, split_name, s, k=k) for s in spec["variants"]]
        candidates = rrf_fuse(runs, keep_k=k, rrf_k=int(spec.get("rrf_k", BM25_RRF_K)))
    else:
        candidates = retrieve_single_bm25(claims_dict, spec, k=k)

    with open(cache_file, "wb") as f:
        pickle.dump(candidates, f)
    return candidates


def strip_scores(candidate_cache, k):
    return {cid: [eid for eid, _ in pairs[:k]] for cid, pairs in candidate_cache.items()}


def candidate_pool_stats(candidate_cache, gold_claims):
    recalls = []
    any_hit = []
    perfect = []
    for cid, claim in gold_claims.items():
        cand_set = {eid for eid, _ in candidate_cache[cid]}
        gold = list(claim["evidences"])
        hit = sum(1 for eid in gold if eid in cand_set)
        rec = hit / max(1, len(gold))
        recalls.append(rec)
        any_hit.append(hit > 0)
        perfect.append(hit == len(gold))
    return {
        "mean_recall_at_500": float(np.mean(recalls)),
        "p50_recall_at_500": float(np.percentile(recalls, 50)),
        "p10_recall_at_500": float(np.percentile(recalls, 10)),
        "any_hit_rate": float(np.mean(any_hit)),
        "perfect_rate": float(np.mean(perfect)),
    }


In [7]:

# Stage 1: cheap BM25-only parameter sweep.
# This does not decide final retrieval F by itself; it selects promising candidate pools for reranking.
rough_rows = []
for spec in BM25_PARAM_GRID:
    t0 = time.time()
    dev_candidates = compute_or_load_candidates(dev_claims, "dev", spec, BM25_CANDIDATE_K)
    stats = candidate_pool_stats(dev_candidates, dev_claims)
    row = {
        "spec_name": spec_name(spec),
        "stopwords": "None" if spec.get("stopwords") is None else spec.get("stopwords"),
        "k1": spec.get("k1", 1.5),
        "b": spec.get("b", 0.75),
        **stats,
        "time_sec": time.time() - t0,
    }
    rough_rows.append(row)

rough_df = pd.DataFrame(rough_rows).sort_values(
    ["mean_recall_at_500", "perfect_rate", "any_hit_rate"], ascending=False
).reset_index(drop=True)
display(rough_df)
rough_path = OUTPUT_DIR / "retrieval_branch_stage1_bm25_sweep.csv"
rough_df.to_csv(rough_path, index=False)
print("Saved:", rough_path)


Computing dev candidates for baseline_stop_en_stop-en_k11.5_b0.75 ...
Tokenizing/building BM25 index: baseline_stop_en_stop-en_k11.5_b0.75


Built baseline_stop_en_stop-en_k11.5_b0.75 in 23.2s


Computing dev candidates for low_b_stop_en_stop-en_k11.5_b0.3 ...
Tokenizing/building BM25 index: low_b_stop_en_stop-en_k11.5_b0.3


Built low_b_stop_en_stop-en_k11.5_b0.3 in 23.4s


Computing dev candidates for mid_b_stop_en_stop-en_k11.5_b0.5 ...
Tokenizing/building BM25 index: mid_b_stop_en_stop-en_k11.5_b0.5


...

Built baseline_no_stop_nostop_k11.5_b0.75 in 27.5s


Computing dev candidates for low_b_no_stop_nostop_k11.5_b0.3 ...
Tokenizing/building BM25 index: low_b_no_stop_nostop_k11.5_b0.3


Built low_b_no_stop_nostop_k11.5_b0.3 in 29.9s


,spec_name,stopwords,k1,b,mean_recall_at_500,p50_recall_at_500,p10_recall_at_500,any_hit_rate,perfect_rate,time_sec
0,mid_b_stop_en_stop-en_k11.5_b0.5,en,1.5,0.50,0.650216,0.750000,0.0,0.870130,0.376623,28.980374
1,mid_k1_low_b_stop_en_stop-en_k11.2_b0.3,en,1.2,0.30,0.648052,0.750000,0.0,0.863636,0.370130,30.663187
2,low_k1_low_b_stop_en_stop-en_k10.8_b0.3,en,0.8,0.30,0.648052,0.750000,0.0,0.863636,0.376623,29.890144
3,low_b_stop_en_stop-en_k11.5_b0.3,en,1.5,0.30,0.646753,0.750000,0.0,0.863636,0.370130,27.403623
4,low_k1_stop_en_stop-en_k10.8_b0.75,en,0.8,0.75,0.644156,0.750000,0.0,0.870130,0.376623,29.906607
5,mid_k1_stop_en_stop-en_k11.2_b0.75,en,1.2,0.75,0.632792,0.666667,0.0,0.870130,0.370130,29.698172
6,baseline_no_stop_nostop_k11.5_b0.75,None,1.5,0.75,0.620130,0.633333,0.0,0.870130,0.357143,29.661959
7,baseline_stop_en_stop-en_k11.5_b0.75,en,1.5,0.75,0.617857,0.666667,0.0,0.857143,0.357143,26.790775
8,low_b_no_stop_nostop_k11.5_b0.3,None,1.5,0.30,0.612229,0.633333,0.0,0.850649,0.357143,32.384459
9,high_k1_stop_en_stop-en_k12.0_b0.75,en,2.0,0.75,0.600433,0.600000,0.0,0.837662,0.331169,30.064174


Saved: outputs_notebook_retrieval_branch\retrieval_branch_stage1_bm25_sweep.csv


## Stage 2: CE reranker comparison over promising coarse candidate pools

In [8]:

from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

reranker_tokenizer = AutoTokenizer.from_pretrained(RERANKER_MODEL_NAME)


def make_reranker_stage_specs():
    baseline = BM25_PARAM_GRID[0]
    by_name = {spec_name(baseline): baseline}

    # Add top-N single BM25 configs from the cheap stage.
    for name in rough_df["spec_name"].head(RERANKER_STAGE_TOP_SINGLE_CONFIGS):
        for spec in BM25_PARAM_GRID:
            if spec_name(spec) == name:
                by_name[name] = spec
                break

    # Add RRF unions. These test variant union without blindly increasing candidate_k.
    top_specs = []
    for name in rough_df["spec_name"].head(3):
        for spec in BM25_PARAM_GRID:
            if spec_name(spec) == name:
                top_specs.append(spec)
                break
    if len(top_specs) >= 2:
        by_name["rrf_top3_single_configs"] = {
            "type": "rrf_union",
            "name": "top3_single_configs",
            "rrf_k": BM25_RRF_K,
            "variants": top_specs,
        }

    # Explicitly test baseline + no-stop + best low-b if available.
    no_stop = next((s for s in BM25_PARAM_GRID if s["name"] == "baseline_no_stop"), None)
    low_b = next((s for s in BM25_PARAM_GRID if s["name"] == "low_b_stop_en"), None)
    variants = [s for s in [baseline, no_stop, low_b] if s is not None]
    if len(variants) >= 2:
        by_name["rrf_baseline_no_stop_low_b"] = {
            "type": "rrf_union",
            "name": "baseline_no_stop_low_b",
            "rrf_k": BM25_RRF_K,
            "variants": variants,
        }

    return list(by_name.values())


class RerankerPairDataset(Dataset):
    def __init__(self, examples, claims_dict, evidence_dict, tokenizer, max_len=RERANKER_MAX_LEN):
        self.examples = examples
        self.claims = claims_dict
        self.evidence = evidence_dict
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        enc = self.tokenizer(
            self.claims[ex["cid"]]["claim_text"],
            self.evidence[ex["eid"]],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(float(ex["label"]), dtype=torch.float32)
        return item


def build_reranker_examples(claims_dict, bm25_candidates, negatives_per_positive=NEGATIVES_PER_POSITIVE):
    examples = []
    rng = random.Random(SEED)
    for cid, claim in claims_dict.items():
        gold = [eid for eid in claim["evidences"] if eid in evidence]
        gold_set = set(gold)
        for eid in gold:
            examples.append({"cid": cid, "eid": eid, "label": 1.0})

        candidate_negs = []
        seen = set()
        for eid, _ in bm25_candidates[cid][:HARD_NEGATIVE_POOL]:
            if eid in gold_set or eid in seen or eid not in evidence:
                continue
            candidate_negs.append(eid)
            seen.add(eid)

        needed = negatives_per_positive * max(1, len(gold))
        if len(candidate_negs) > needed:
            candidate_negs = rng.sample(candidate_negs, needed)
        for eid in candidate_negs:
            examples.append({"cid": cid, "eid": eid, "label": 0.0})
    rng.shuffle(examples)
    return examples


def load_fresh_reranker():
    model = AutoModelForSequenceClassification.from_pretrained(
        RERANKER_MODEL_NAME,
        num_labels=1,
        ignore_mismatched_sizes=True,
    ).to(DEVICE)
    return model


@torch.no_grad()
def score_candidates_with_model(model, claims_dict, candidates, split_name, spec_tag, allow_cache=True):
    score_cache_file = CACHE_DIR / f"{split_name}_{spec_tag}_ce_scores.pkl"
    if allow_cache and score_cache_file.exists() and not FORCE_RESCORE_CE:
        print("Loading cached CE scores:", score_cache_file)
        with open(score_cache_file, "rb") as f:
            return pickle.load(f)

    model.eval()
    score_cache = {}
    print(f"Scoring {split_name} candidates with CE reranker [{spec_tag}] ...")
    for cid, claim in tqdm(list(claims_dict.items())):
        cand_pairs = candidates[cid]
        cand_eids = [eid for eid, _ in cand_pairs]
        bm25_scores = np.array([score for _, score in cand_pairs], dtype=np.float32)
        logits_all = []
        for start in range(0, len(cand_eids), RERANKER_EVAL_BATCH_SIZE):
            batch_eids = cand_eids[start:start + RERANKER_EVAL_BATCH_SIZE]
            enc = reranker_tokenizer(
                [claim["claim_text"]] * len(batch_eids),
                [evidence[eid] for eid in batch_eids],
                truncation=True,
                padding=True,
                max_length=RERANKER_MAX_LEN,
                return_tensors="pt",
            ).to(DEVICE)
            logits = model(**enc).logits.squeeze(-1)
            logits_all.extend(logits.detach().cpu().tolist())
        logits_np = np.array(logits_all, dtype=np.float32)
        score_cache[cid] = {
            "eids": cand_eids,
            "bm25": bm25_scores,
            "ce_logit": logits_np,
            "ce_prob": 1.0 / (1.0 + np.exp(-logits_np)),
        }

    with open(score_cache_file, "wb") as f:
        pickle.dump(score_cache, f)
    print("Cached CE scores:", score_cache_file)
    return score_cache


def select_fixed_k_ce(score_cache, k):
    out = {}
    for cid, entry in score_cache.items():
        order = np.argsort(-entry["ce_logit"])[:k]
        out[cid] = [entry["eids"][int(i)] for i in order]
    return out


def select_dynamic_threshold_ce(score_cache, threshold, max_k=MAX_FINAL_K, min_k=MIN_FINAL_K):
    out = {}
    for cid, entry in score_cache.items():
        probs = entry["ce_prob"]
        order = np.argsort(-probs)
        selected = [int(i) for i in order[:max_k] if probs[int(i)] >= threshold]
        if len(selected) < min_k:
            selected = [int(i) for i in order[:min_k]]
        out[cid] = [entry["eids"][i] for i in selected[:max_k]]
    return out


def select_relative_logit_ce(score_cache, delta, max_k=MAX_FINAL_K, min_k=MIN_FINAL_K):
    out = {}
    for cid, entry in score_cache.items():
        logits = entry["ce_logit"]
        order = np.argsort(-logits)
        top = float(logits[int(order[0])])
        selected = [int(i) for i in order[:max_k] if top - float(logits[int(i)]) <= delta]
        if len(selected) < min_k:
            selected = [int(i) for i in order[:min_k]]
        out[cid] = [entry["eids"][i] for i in selected[:max_k]]
    return out


def tune_retrieval_from_ce_scores(dev_score_cache):
    rows = []
    for k in FIXED_K_GRID:
        retr = select_fixed_k_ce(dev_score_cache, k=k)
        rows.append({"mode": "fixed_k", "k": k, "threshold": np.nan, "delta": np.nan,
                     "retrieval_F": evaluate_retrieval_only(retr, dev_claims),
                     "avg_pred_evidence": np.mean([len(v) for v in retr.values()])})
    for threshold in THRESHOLD_GRID:
        retr = select_dynamic_threshold_ce(dev_score_cache, threshold=threshold)
        rows.append({"mode": "dynamic_threshold", "k": np.nan, "threshold": threshold, "delta": np.nan,
                     "retrieval_F": evaluate_retrieval_only(retr, dev_claims),
                     "avg_pred_evidence": np.mean([len(v) for v in retr.values()])})
    for delta in RELATIVE_LOGIT_DELTA_GRID:
        retr = select_relative_logit_ce(dev_score_cache, delta=delta)
        rows.append({"mode": "relative_logit", "k": np.nan, "threshold": np.nan, "delta": delta,
                     "retrieval_F": evaluate_retrieval_only(retr, dev_claims),
                     "avg_pred_evidence": np.mean([len(v) for v in retr.values()])})
    return pd.DataFrame(rows).sort_values("retrieval_F", ascending=False).reset_index(drop=True)


def choose_final_retrieval_setting(results_df):
    results_df = results_df.sort_values("retrieval_F", ascending=False).reset_index(drop=True)
    best = results_df.iloc[0]
    if FINAL_RETRIEVAL_POLICY == "best_dev":
        chosen = best
    elif FINAL_RETRIEVAL_POLICY in {"fixed_k", "dynamic_threshold", "relative_logit"}:
        chosen = results_df[results_df["mode"] == FINAL_RETRIEVAL_POLICY].iloc[0]
    elif FINAL_RETRIEVAL_POLICY == "prefer_dynamic":
        dynamic = results_df[results_df["mode"].isin(["dynamic_threshold", "relative_logit"])]
        eligible = dynamic[dynamic["retrieval_F"] >= float(best["retrieval_F"]) - DYNAMIC_RETRIEVAL_TOLERANCE]
        chosen = eligible.iloc[0] if not eligible.empty else best
    else:
        raise ValueError(FINAL_RETRIEVAL_POLICY)
    return chosen.to_dict()


def apply_retrieval_setting(score_cache, row):
    if row["mode"] == "fixed_k":
        return select_fixed_k_ce(score_cache, int(row["k"]))
    if row["mode"] == "dynamic_threshold":
        return select_dynamic_threshold_ce(score_cache, float(row["threshold"]))
    if row["mode"] == "relative_logit":
        return select_relative_logit_ce(score_cache, float(row["delta"]))
    raise ValueError(row)


In [9]:

# Stage 2: train/evaluate one reranker per promising candidate-pool strategy.
# This is intentionally long, but it gives a controlled comparison in one notebook run.

def train_or_load_reranker_for_spec(spec, train_candidates, dev_candidates):
    tag = f"{CACHE_VERSION}_{spec_name(spec)}_{safe_name(RERANKER_MODEL_NAME)}_top{BM25_CANDIDATE_K}_seed{SEED}"
    ckpt_path = OUTPUT_DIR / f"reranker_best_{tag}.pt"

    model = load_fresh_reranker()
    best_row = None
    freshly_trained = False

    if SAVE_MODEL_CHECKPOINTS and ckpt_path.exists() and not FORCE_RERANKER_RETRAIN:
        print("Loading cached reranker checkpoint:", ckpt_path)
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(ckpt["model_state"])
        best_row = ckpt.get("best_row")
        return model, best_row, tag, freshly_trained

    freshly_trained = True
    examples = build_reranker_examples(train_claims, train_candidates)
    pos = sum(1 for x in examples if x["label"] == 1.0)
    neg = len(examples) - pos
    print(f"Reranker examples [{spec_name(spec)}]: {len(examples):,} | positives={pos:,} negatives={neg:,}")

    ds = RerankerPairDataset(examples, train_claims, evidence, reranker_tokenizer)
    loader = DataLoader(ds, batch_size=RERANKER_BATCH_SIZE, shuffle=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=RERANKER_LR, weight_decay=WEIGHT_DECAY)
    total_steps = len(loader) * RERANKER_EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
    loss_fn = nn.BCEWithLogitsLoss()

    best_state = None
    best_F = -1.0
    for epoch in range(1, RERANKER_EPOCHS + 1):
        model.train()
        losses = []
        t0 = time.time()
        for batch in tqdm(loader, desc=f"Reranker {spec_name(spec)} epoch {epoch}/{RERANKER_EPOCHS}"):
            labels = batch.pop("labels").to(DEVICE)
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            logits = model(**batch).logits.squeeze(-1)
            loss = loss_fn(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            losses.append(float(loss.item()))

        dev_scores_tmp = score_candidates_with_model(model, dev_claims, dev_candidates, f"dev_epoch{epoch}", tag, allow_cache=False)
        results_tmp = tune_retrieval_from_ce_scores(dev_scores_tmp)
        row = choose_final_retrieval_setting(results_tmp)
        print(f"Epoch {epoch}: loss={np.mean(losses):.4f} | chosen retrieval_F={row['retrieval_F']:.4f} | row={row} | time={time.time()-t0:.1f}s")
        if row["retrieval_F"] > best_F:
            best_F = float(row["retrieval_F"])
            best_row = row
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    if SAVE_MODEL_CHECKPOINTS:
        torch.save({"model_state": model.state_dict(), "best_row": best_row, "spec": spec}, ckpt_path)
        print("Saved reranker checkpoint:", ckpt_path)
    return model, best_row, tag, freshly_trained


retrieval_stage_specs = make_reranker_stage_specs()
print("Reranker-stage candidate strategies:")
for s in retrieval_stage_specs:
    print(" -", spec_name(s))

retrieval_results_rows = []
best_overall = None

if RUN_RERANKER_STAGE:
    for spec in retrieval_stage_specs:
        set_seed(SEED)
        spec_tag_short = spec_name(spec)
        print("\n" + "=" * 80)
        print("Retrieval experiment:", spec_tag_short)
        print("=" * 80)

        train_candidates = compute_or_load_candidates(train_claims, "train", spec, BM25_CANDIDATE_K)
        dev_candidates = compute_or_load_candidates(dev_claims, "dev", spec, BM25_CANDIDATE_K)
        pool_stats = candidate_pool_stats(dev_candidates, dev_claims)

        model, epoch_best_row, tag, freshly_trained = train_or_load_reranker_for_spec(spec, train_candidates, dev_candidates)
        dev_scores = score_candidates_with_model(model, dev_claims, dev_candidates, "dev_final", tag, allow_cache=not freshly_trained)
        selector_df = tune_retrieval_from_ce_scores(dev_scores)
        display(selector_df.head(10))
        chosen = choose_final_retrieval_setting(selector_df)
        dev_retrieval = apply_retrieval_setting(dev_scores, chosen)
        validate_retrieval_coverage(dev_claims, dev_retrieval, split_name=f"dev/{spec_tag_short}")

        row = {
            "spec_name": spec_tag_short,
            "candidate_type": spec.get("type", "single"),
            **pool_stats,
            "chosen_mode": chosen["mode"],
            "chosen_k": chosen.get("k", np.nan),
            "chosen_threshold": chosen.get("threshold", np.nan),
            "chosen_delta": chosen.get("delta", np.nan),
            "dev_retrieval_F": float(chosen["retrieval_F"]),
            "avg_pred_evidence": float(chosen["avg_pred_evidence"]),
        }
        retrieval_results_rows.append(row)
        results_df = pd.DataFrame(retrieval_results_rows).sort_values("dev_retrieval_F", ascending=False).reset_index(drop=True)
        results_path = OUTPUT_DIR / "retrieval_branch_stage2_reranker_results.csv"
        results_df.to_csv(results_path, index=False)
        display(results_df)
        print("Saved:", results_path)

        if best_overall is None or row["dev_retrieval_F"] > best_overall["row"]["dev_retrieval_F"]:
            best_overall = {"row": row, "spec": spec, "chosen": chosen, "tag": tag, "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}}

        del model
        torch.cuda.empty_cache()
        gc.collect()

print("Best retrieval branch result:")
if best_overall is not None:
    print(best_overall["row"])


Reranker-stage candidate strategies:
 - baseline_stop_en_stop-en_k11.5_b0.75
 - mid_b_stop_en_stop-en_k11.5_b0.5
 - mid_k1_low_b_stop_en_stop-en_k11.2_b0.3
 - low_k1_low_b_stop_en_stop-en_k10.8_b0.3
 - rrf_top3_single_configs
 - rrf_baseline_no_stop_low_b

Retrieval experiment: baseline_stop_en_stop-en_k11.5_b0.75
Computing train candidates for baseline_stop_en_stop-en_k11.5_b0.75 ...
Loading BM25 index: outputs_notebook_retrieval_branch\cache\bm25s_index_baseline_stop_en_stop-en_k11.5_b0.75_1208827docs


Loading cached candidates: outputs_notebook_retrieval_branch\cache\dev_candidates_baseline_stop_en_stop-en_k11.5_b0.75_top500.pkl


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 12678.91it/s]


Reranker examples [baseline_stop_en_stop-en_k11.5_b0.75]: 20,610 | positives=4,122 negatives=16,488


Reranker baseline_stop_en_stop-en_k11.5_b0.75 epoch 1/3: 100%|██████████| 645/645 [01:46<00:00,  6.08it/s]


...

100%|██████████| 154/154 [00:59<00:00,  2.57it/s]

Cached CE scores: outputs_notebook_retrieval_branch\cache\dev_final_retrieval_branch_v5_from_v3_top500_rrf_baseline_no_stop_low_b_cross-encoder-ms-marco-MiniLM-L-6-v2_top500_seed42_ce_scores.pkl


,mode,k,threshold,delta,retrieval_F,avg_pred_evidence
0,relative_logit,NaN,NaN,1.50,0.217754,4.279221
1,relative_logit,NaN,NaN,2.00,0.216625,4.636364
2,relative_logit,NaN,NaN,1.00,0.207395,3.564935
3,relative_logit,NaN,NaN,3.00,0.203365,4.837662
4,relative_logit,NaN,NaN,0.75,0.203169,3.064935
5,fixed_k,4.0,NaN,NaN,0.202958,4.000000
6,fixed_k,2.0,NaN,NaN,0.194929,2.000000
7,fixed_k,3.0,NaN,NaN,0.194712,3.000000
8,dynamic_threshold,NaN,0.48,NaN,0.193414,4.259740
9,dynamic_threshold,NaN,0.34,NaN,0.192867,4.694805


,spec_name,candidate_type,mean_recall_at_500,p50_recall_at_500,p10_recall_at_500,any_hit_rate,perfect_rate,chosen_mode,chosen_k,chosen_threshold,chosen_delta,dev_retrieval_F,avg_pred_evidence
0,mid_b_stop_en_stop-en_k11.5_b0.5,single,0.650216,0.750000,0.0,0.870130,0.376623,relative_logit,NaN,NaN,1.5,0.227850,4.331169
1,low_k1_low_b_stop_en_stop-en_k10.8_b0.3,single,0.648052,0.750000,0.0,0.863636,0.376623,relative_logit,NaN,NaN,1.0,0.225155,3.337662
2,mid_k1_low_b_stop_en_stop-en_k11.2_b0.3,single,0.648052,0.750000,0.0,0.863636,0.370130,relative_logit,NaN,NaN,1.5,0.223114,4.305195
3,rrf_top3_single_configs,rrf_union,0.649675,0.750000,0.0,0.870130,0.370130,relative_logit,NaN,NaN,1.5,0.220712,4.363636
4,rrf_baseline_no_stop_low_b,rrf_union,0.644156,0.750000,0.0,0.870130,0.376623,relative_logit,NaN,NaN,1.5,0.217754,4.279221
5,baseline_stop_en_stop-en_k11.5_b0.75,single,0.617857,0.666667,0.0,0.857143,0.357143,relative_logit,NaN,NaN,1.5,0.210560,4.246753


Saved: outputs_notebook_retrieval_branch\retrieval_branch_stage2_reranker_results.csv
Best retrieval branch result:
{'spec_name': 'mid_b_stop_en_stop-en_k11.5_b0.5', 'candidate_type': 'single', 'mean_recall_at_500': 0.6502164502164501, 'p50_recall_at_500': 0.75, 'p10_recall_at_500': 0.0, 'any_hit_rate': 0.8701298701298701, 'perfect_rate': 0.37662337662337664, 'chosen_mode': 'relative_logit', 'chosen_k': nan, 'chosen_threshold': nan, 'chosen_delta': 1.5, 'dev_retrieval_F': 0.22784992784992786, 'avg_pred_evidence': 4.3311688311688314}


## Export best retrieval artifacts

In [10]:

# Export best retrieval artifacts for later connection to the classifier branch.
# This cell does not inspect test labels; it only creates retrieved evidence lists.
if best_overall is None:
    raise RuntimeError("Run the reranker-stage experiment first.")

best_spec = best_overall["spec"]
best_chosen = best_overall["chosen"]
best_tag = best_overall["tag"]
print("Best spec:", spec_name(best_spec))
print("Best selector:", best_chosen)

best_model = load_fresh_reranker()
best_model.load_state_dict(best_overall["model_state"])

exports = {}
for split_name, claims in [("train", train_claims), ("dev", dev_claims), ("test", test_claims)]:
    candidates = compute_or_load_candidates(claims, split_name, best_spec, BM25_CANDIDATE_K)
    scores = score_candidates_with_model(best_model, claims, candidates, f"{split_name}_best", best_tag, allow_cache=True)
    retrieval = apply_retrieval_setting(scores, best_chosen)
    validate_retrieval_coverage(claims, retrieval, split_name=f"best/{split_name}")
    out_path = OUTPUT_DIR / f"{split_name}_retrieval_best.pkl"
    with open(out_path, "wb") as f:
        pickle.dump(retrieval, f)
    exports[split_name] = str(out_path)
    print("Saved:", out_path)

config_path = OUTPUT_DIR / "best_retrieval_branch_config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump({
        "best_row": best_overall["row"],
        "best_spec_name": spec_name(best_spec),
        "best_spec": best_spec,
        "best_selector": best_chosen,
        "exports": exports,
    }, f, indent=2, ensure_ascii=True)
print("Saved:", config_path)

Best spec: mid_b_stop_en_stop-en_k11.5_b0.5
Best selector: {'mode': 'relative_logit', 'k': nan, 'threshold': nan, 'delta': 1.5, 'retrieval_F': 0.22784992784992786, 'avg_pred_evidence': 4.3311688311688314}


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 5299.15it/s]


Loading cached candidates: outputs_notebook_retrieval_branch\cache\train_candidates_mid_b_stop_en_stop-en_k11.5_b0.5_top500.pkl
Scoring train_best candidates with CE reranker [retrieval_branch_v5_from_v3_top500_mid_b_stop_en_stop-en_k11.5_b0.5_cross-encoder-ms-marco-MiniLM-L-6-v2_top500_seed42] ...


100%|██████████| 1228/1228 [07:25<00:00,  2.76it/s]


Cached CE scores: outputs_notebook_retrieval_branch\cache\train_best_retrieval_branch_v5_from_v3_top500_mid_b_stop_en_stop-en_k11.5_b0.5_cross-encoder-ms-marco-MiniLM-L-6-v2_top500_seed42_ce_scores.pkl
Saved: outputs_notebook_retrieval_branch\train_retrieval_best.pkl
Loading cached candidates: outputs_notebook_retrieval_branch\cache\dev_candidates_mid_b_stop_en_stop-en_k11.5_b0.5_top500.pkl
Scoring dev_best candidates with CE reranker [retrieval_branch_v5_from_v3_top500_mid_b_stop_en_stop-en_k11.5_b0.5_cross-encoder-ms-marco-MiniLM-L-6-v2_top500_seed42] ...


100%|██████████| 154/154 [00:56<00:00,  2.71it/s]


Cached CE scores: outputs_notebook_retrieval_branch\cache\dev_best_retrieval_branch_v5_from_v3_top500_mid_b_stop_en_stop-en_k11.5_b0.5_cross-encoder-ms-marco-MiniLM-L-6-v2_top500_seed42_ce_scores.pkl
Saved: outputs_notebook_retrieval_branch\dev_retrieval_best.pkl
Computing test candidates for mid_b_stop_en_stop-en_k11.5_b0.5 ...
Loading BM25 index: outputs_notebook_retrieval_branch\cache\bm25s_index_mid_b_stop_en_stop-en_k11.5_b0.5_1208827docs


Scoring test_best candidates with CE reranker [retrieval_branch_v5_from_v3_top500_mid_b_stop_en_stop-en_k11.5_b0.5_cross-encoder-ms-marco-MiniLM-L-6-v2_top500_seed42] ...


100%|██████████| 153/153 [00:57<00:00,  2.67it/s]

Cached CE scores: outputs_notebook_retrieval_branch\cache\test_best_retrieval_branch_v5_from_v3_top500_mid_b_stop_en_stop-en_k11.5_b0.5_cross-encoder-ms-marco-MiniLM-L-6-v2_top500_seed42_ce_scores.pkl
Saved: outputs_notebook_retrieval_branch\test_retrieval_best.pkl
Saved: outputs_notebook_retrieval_branch\best_retrieval_branch_config.json
